# PQC IDS: full main-pipeline reproduction

The returned seed-0 C1-SF check passed independently for RF, XGBoost and MLP.
This notebook takes the next step: rerun the **original main pipeline over
all ten splits**, including KS/Holm selection, mutual information, C1, C2,
C3, all five MI thresholds and all four footprints.

**This is a substantial training run: 420 model fits.** Allow the Colab
runtime to continue until the final export. Progress is printed for each fit.
The original code saves its raw-result checkpoint after each completed split;
logs and status are also saved to Drive. A failed run exports partial results
for review. Re-running all cells creates a fresh run, without silently merging
previous checkpoints.

1. Select **Runtime → Run all**.
2. Upload the updated **PQC_IDS_Reproduction_Working_Copy.zip** supplied with
   this notebook, and connect the Drive account containing your CSV files.
3. Return **PQC_IDS_Main_Reproduction_Results.zip**, even if the comparison
   reports a difference.

This is the main pipeline only. Identifier-free B3, later extensions and the
scientific validation tasks remain separate. The new run uses a documented
prospective environment; it does not claim to recover every historical binary
or package version. Original data and locked results are only read.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_main_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one updated working-copy ZIP.")
archive_path = SESSION / "working_copy.zip"
archive_path.write_bytes(uploaded[zip_names[0]])
del uploaded
extracted = SESSION / "package"
with zipfile.ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
    for info in archive.infolist():
        path = PurePosixPath(info.filename)
        if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError("Unexpected archive path: " + info.filename)
    archive.extractall(extracted)
manifest_paths = list(extracted.glob("*/package_manifest.json"))
if len(manifest_paths) != 1:
    raise ValueError("Expected one working-copy package manifest.")
PACKAGE = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text())
listed = set()
for item in manifest["files"]:
    path = (PACKAGE / item["path"]).resolve()
    if not path.is_relative_to(PACKAGE.resolve()):
        raise ValueError("Invalid manifest path.")
    if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
        raise ValueError("Package file does not match the manifest: " + item["path"])
    listed.add(item["path"])
actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
    raise ValueError("The archive contains unlisted files or duplicate manifest records.")
if not (PACKAGE / "scripts/compare_main_reproduction.py").is_file():
    raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
print("Working-copy files verified:", len(listed))
print("Run identifier:", RUN_ID)

## Connect the four CSV files

The default folder is `/content/drive/MyDrive/cicids2018`. Change `DATA_DIR`
only if you moved the files. Their hashes and the prepared matrix must match
the successful seed-0 run before fitting can begin.

In [ ]:
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/pqc_ids_reproduction_checks") / RUN_ID
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=False)
RUN_OUTPUT = DRIVE_OUTPUT / "results"
TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
SUMMARY = {
    "run_id": RUN_ID,
    "status": "preparing_environment",
    "colab_kernel_python": platform.python_version(),
    "package_sha256": sha256(archive_path),
    "package_manifest_sha256": sha256(PACKAGE / "package_manifest.json"),
    "data_directory": str(DATA_DIR),
    "result_directory": str(DRIVE_OUTPUT),
    "scope": "Full ten-split main pipeline only; B3 and later extensions are separate",
    "target": TARGET,
}

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
shutil.copy2(PACKAGE / "environment/main_known_versions.json", DRIVE_OUTPUT / "main_known_versions.json")
shutil.copy2(PACKAGE / "environment/main_requirements.txt", DRIVE_OUTPUT / "main_requirements.txt")
print("Results will be saved to:", DRIVE_OUTPUT)

## Install the declared reproduction environment

The notebook keeps the Python 3.12.13, NumPy 2.0.2, pandas 2.2.2, SciPy 1.16.3,
scikit-learn 1.6.1 and XGBoost 3.2.0 versions from the successful seed-0 run.
The full pipeline also needs statsmodels. **Statsmodels 0.15.0 is a prospective
pin**, taken from the returned readiness record; it is not presented as the
version used for the original July experiments. All 21 resolved dependencies
are pinned and recorded.

A separate runtime is installed, so your Colab kernel keeps its own packages.
The [statsmodels release page](https://pypi.org/project/statsmodels/0.15.0/)
and uv's [environment documentation](https://docs.astral.sh/uv/pip/environments/)
describe the packages and setup mechanism. Local import and Holm checks passed;
full training on your CSVs is the purpose of this notebook.

In [ ]:
RUN_ENV = os.environ.copy()
for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
    RUN_ENV.pop(name, None)
RUN_ENV["PYTHONNOUSERSITE"] = "1"
RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

def run_logged(command, log_name, env):
    with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
        log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
        log.flush()
        with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                              text=True, bufsize=1, env=env) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return process.wait()

ENVIRONMENT_READY = False
VENV = SESSION / "main_environment"
FIT_PYTHON = VENV / "bin/python"
try:
    tooling = SESSION / "uv_tooling"
    bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                 "--no-deps", "uv==0.12.15"]
    if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
        raise RuntimeError("Could not install the pinned uv tool.")
    uv_env = RUN_ENV.copy()
    uv_env["PYTHONPATH"] = str(tooling)
    uv = [sys.executable, "-m", "uv"]
    create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
    if run_logged(create, "02_python_setup.log", uv_env) != 0:
        raise RuntimeError("Could not create the requested Python runtime.")
    install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                    "-r", PACKAGE / "environment/main_requirements.txt"]
    if run_logged(install, "03_packages_install.log", uv_env) != 0:
        raise RuntimeError("Could not install the declared package versions.")
    ENVIRONMENT_READY = True
    SUMMARY["status"] = "environment_installed"
except Exception as error:
    SUMMARY["status"] = "environment_setup_failed"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
    print("Setup failed:", error)
finally:
    save_summary()
print("Environment ready:", ENVIRONMENT_READY)

## Run all ten main splits and compare against the locked raw results

The original source code, model settings, fitting order, transformations,
thresholds and footprint conditions are preserved. New wrapper code adds
input/matrix identity checks, progress records and a comparison report.

The comparison checks every field in the saved main raw JSON. Missing splits,
features, models or conditions require review. Feature lists and counts must
match exactly. Numerical rules are fixed before the run:

| Field | Absolute tolerance |
|---|---:|
| Weighted/per-class F1, including the MI-sweep F1 scores | 1e-12 |
| Weighted multiclass ROC-AUC | 5e-5 |
| MI estimates | 1e-8 |
| KS statistic | 1e-12 |

These are operational reproduction tolerances, not the paper's equivalence
margins. All differences are saved. The original main raw file has no
prediction arrays, so this comparison cannot assert identical predictions.
A completed training run and a passing comparison are separate checks.

In [ ]:
if ENVIRONMENT_READY:
    try:
        SUMMARY["status"] = "running_main"
        save_summary()
        seed0_reference = PACKAGE / "reference/seed0_reproduction_20260920/results"
        command = [FIT_PYTHON, "-u", PACKAGE / "scripts/run_experiment.py",
                   "--stage", "train", "--variant", "main",
                   "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                   "--cache-dir", SESSION / "input_cache",
                   "--expected-environment", PACKAGE / "environment/main_known_versions.json",
                   "--expected-inputs", seed0_reference / "input_files.json",
                   "--expected-prepared", seed0_reference / "seed0_status.json"]
        returncode = run_logged(command, "04_main_run.log", RUN_ENV)
        SUMMARY["main_training_returncode"] = returncode
        status_path = RUN_OUTPUT / "run_record.json"
        run_record = json.loads(status_path.read_text()) if status_path.is_file() else {}
        SUMMARY["training_status"] = run_record.get("status", "no_run_record")
        SUMMARY["fits_completed"] = run_record.get("fits_completed", 0)
        SUMMARY["expected_fits"] = 420
        SUMMARY["status"] = "main_training_completed" if returncode == 0 else "main_training_failed"
        raw_path = RUN_OUTPUT / "revised_pipeline_raw.json"
        if raw_path.is_file():
            comparison_output = DRIVE_OUTPUT / "comparison"
            compare_command = [FIT_PYTHON, "-u", PACKAGE / "scripts/compare_main_reproduction.py",
                               "--actual", raw_path, "--output", comparison_output]
            comparison_returncode = run_logged(compare_command, "05_main_comparison.log", RUN_ENV)
            SUMMARY["comparison_returncode"] = comparison_returncode
            successful_run = (returncode == 0 and run_record.get("status") == "completed"
                              and run_record.get("fits_completed") == 420
                              and run_record.get("split_computations_completed") == list(range(10)))
            SUMMARY["status"] = "passed_main_reproduction" if successful_run and comparison_returncode == 0 else "needs_review"
            means = comparison_output / "main_headline_means.csv"
            if means.is_file():
                print(means.read_text())
        print("\nMain run status:", SUMMARY["status"])
    except Exception as error:
        SUMMARY["status"] = "runner_exception"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "runner_traceback.txt").write_text(traceback.format_exc())
        print("Runner failed:", error)
    finally:
        save_summary()
else:
    print("Training skipped because the declared environment was not installed.")

## Export results, differences and logs

Run this cell even after a failure. It exports the new raw/summary results,
comparison tables, runtime and data identity records, warnings and logs.
Input CSVs and installation caches stay outside the result ZIP.

The ZIP is also saved in the new Drive result folder. Return
**PQC_IDS_Main_Reproduction_Results.zip** for review. Preserve any differences;
do not change thresholds or overwrite the original results to force a match.

In [ ]:
SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
save_summary()
result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*")
                      if p.is_file() and p.name not in ("PQC_IDS_Main_Reproduction_Results.zip", "result_manifest.json"))
manifest = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
             "sha256": sha256(p)} for p in result_files]
manifest_file = DRIVE_OUTPUT / "result_manifest.json"
manifest_file.write_text(json.dumps({"run_id": RUN_ID, "files": manifest}, indent=2) + "\n")
result_files.append(manifest_file)
result_zip = SESSION / "PQC_IDS_Main_Reproduction_Results.zip"
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in result_files:
        archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
with zipfile.ZipFile(result_zip) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("The result archive failed its integrity check.")
saved_zip = DRIVE_OUTPUT / result_zip.name
shutil.copy2(result_zip, saved_zip)
print("Final status:", SUMMARY["status"])
print("Results saved:", saved_zip)
print("ZIP SHA-256:", sha256(result_zip))
files.download(str(result_zip))